In [0]:
def read_csv_df(spark,path,header=True,infer_schema=True,sep=","):
    return (spark.read
        .option("header", header)
        .option("inferSchema", infer_schema)
        .option("sep", sep)
        .csv(path))

df1=read_csv_df(spark,'/Volumes/logistics_catalog_assign/landing_zone/landing_vol/logistics_source1.txt',True,True,',')
df1.display()
df2=read_csv_df(spark,'/Volumes/logistics_catalog_assign/landing_zone/landing_vol/logistics_source2.txt',True,True,',')
df2.display()

In [0]:
from pyspark.sql.functions import concat,lit
def read_json(apark,path,mline):
    df = spark.read.json(path, multiLine=mline)
    return df

jsondf=read_json(spark,'/Volumes/logistics_catalog_assign/landing_zone/landing_vol/logistics_shipment_detail_3000.json',True)
jsondf.display()

In [0]:
#Frequently Used Simple Joins (Inner, Left)
df3 = df1.join(df2, df1.shipment_id == df2.shipment_id, 'inner')
df3 = df3.select(
    df1['first_name'],
    df1['last_name'],
    df1['age'],
    df1['role'],
    df2['hub_location'],
    df2['vehicle_type'],
    df2['shipment_id'].alias('shipment_id_df2')
)
final_df = jsondf.join(df3, jsondf.shipment_id == df3.shipment_id_df2, 'inner')
final_df.display()

In [0]:
finals_df = jsondf.join(df2, jsondf.shipment_id == df2.shipment_id, 'left').filter(jsondf.shipment_id.isNull())
finals_df.display()

In [0]:
finals_df = jsondf.join(df1, jsondf.shipment_id == df1.shipment_id, 'left').filter(jsondf.shipment_id.isNull())
finals_df.display()

In [0]:
#Infrequent Simple Joins (Self, Right, Full, Cartesian)
#Self Join (Peer Finding):
df3= df3.join(df3, df3.hub_location == df3.hub_location, 'inner').filter(df3.shipment_id != df3.shipment_id)
jsondf.display()



In [0]:
#Right Join (Orphan Data Check):
from pyspark.sql.functions import col

df3_a = df3.alias('a')
df3_b = df3.alias('b')
joinright_df = df3_a.join(df3_b, col('a.hub_location') == col('b.hub_location'), 'right').filter(col('a.shipment_id_df2') != col('b.shipment_id_df2'))
joinright_df.display()

In [0]:
#Lookup
masterdf=read_csv_df(spark,'/Volumes/logistics_catalog_assign/landing_zone/landing_vol/Master_City_List.csv',True,True,',')
masterdf.display()



In [0]:
#Lookup
from pyspark.sql.functions import col
# Alias both DataFrames
df3_alias = df3.alias("d3")
masterdf_alias = masterdf.alias("m")

# Perform the join using qualified column names
locationdf = df3_alias.join(
    masterdf_alias,
    col("d3.hub_location") == col("m.city_name"),
    "inner"
)
locationdf.display()

In [0]:
#Grouping & Aggregations (Advanced)
finalized_df = jsondf.join(df2, jsondf.shipment_id == df2.shipment_id, 'inner')
finalized_df = finalized_df.select(df2['vehicle_type'], finalized_df['shipment_cost'])
finalized_df = finalized_df.groupBy('vehicle_type').agg({'shipment_cost': 'sum'})
finalized_df.display()

In [0]:
#Grouping & Aggregations (Advanced)
finalized_df = jsondf.join(df2, jsondf.shipment_id == df2.shipment_id, 'inner')
finalized_df = finalized_df.select(df2['hub_location'], finalized_df['shipment_cost'])
finalized_df = finalized_df.groupBy('hub_location').agg({'shipment_cost': 'sum'})
finalized_df.display()

In [0]:
#Grouping & Aggregations (Advanced)
finalized_df = jsondf.join(df2, jsondf.shipment_id == df2.shipment_id, 'inner')
finalized_df = finalized_df.select(df2['hub_location'], df2['vehicle_type'], finalized_df['shipment_cost'])
finalized_df = finalized_df.groupBy('hub_location','vehicle_type').agg({'shipment_cost': 'sum'})
finalized_df.display()
finalized_df = finalized_df.agg({'sum(shipment_cost)': 'avg'})
finalized_df.display()


In [0]:
#Set Operations
df4 = df1.unionAll(df2)
df4.display()